In [1]:
#Best model + function for results
import pandas as pd
import sys
import os
import warnings
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

sys.path.append(os.path.abspath('..'))
from preprocessing import train_clean

warnings.filterwarnings('ignore')

X = train_clean.drop(columns=['Survived'])
y = train_clean['Survived']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def create_pipe(model):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])

best_estimators = {
    'CatBoost': create_pipe(CatBoostClassifier(iterations=150, learning_rate=0.05, depth=5, random_state=42, verbose=0)),
    'LightGBM': create_pipe(LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42, verbose=-1)),
    'XGBoost': create_pipe(XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42, eval_metric='logloss')),
    'RandomForest': create_pipe(RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)),
    'LogisticRegression': create_pipe(LogisticRegression(C=1.0, penalty='l2', solver='liblinear', random_state=42)),
    'SVC': create_pipe(SVC(C=1.0, kernel='rbf', probability=True, random_state=42))
}

top_models = [
    ('catboost', best_estimators['CatBoost']),
    ('lgbm', best_estimators['LightGBM']),
    ('xgb', best_estimators['XGBoost']),
    ('rf', best_estimators['RandomForest']),
    ('logreg', best_estimators['LogisticRegression']),
    ('svc', best_estimators['SVC']),
]

ensemble_results = []
def ensemble_res(name, model):
    scores = cross_val_score(model, X, y, cv = cv, scoring = 'accuracy', n_jobs = 1)
    mean_score = scores.mean()
    std_score = scores.std()
    
    ensemble_results.append({
        'Ensemble': name,
        'Accuracy': mean_score,
        'Std': std_score
    })
    print(f'[{name}] Accuracy: {mean_score:.4f} (+/- {std_score:.4f})')

In [2]:
#Voting_classifier
from sklearn.ensemble import VotingClassifier

voting_hard = VotingClassifier(estimators = top_models, voting = 'hard')
ensemble_res('Voting hard', voting_hard)

voting_soft = VotingClassifier(estimators = top_models, voting = 'soft')
ensemble_res('Voting soft', voting_soft)

[Voting hard] Accuracy: 0.8765 (+/- 0.0143)
[Voting soft] Accuracy: 0.8754 (+/- 0.0121)


In [3]:
#Weighted_averaging
weighted_boosters = VotingClassifier(
    estimators = [
        ('catboost', best_estimators['CatBoost']),
        ('lgbm', best_estimators['LightGBM']),
        ('xgb', best_estimators['XGBoost'])
    ],
    voting = 'soft',
    weights = [3, 2, 2]
)

ensemble_res('Weighted averaging Top Boosters', weighted_boosters)

[Weighted averaging Top Boosters] Accuracy: 0.8743 (+/- 0.0150)


In [4]:
#Stacking_classifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import RidgeClassifier

stacking_ridge = StackingClassifier(
    estimators = top_models,
    final_estimator = RidgeClassifier(random_state = 42),
    cv = cv
)

ensemble_res('Stacking Ridge Meta-model', stacking_ridge)

[Stacking Ridge Meta-model] Accuracy: 0.8743 (+/- 0.0132)


In [5]:
#Bagging_classifier
from sklearn.ensemble import BaggingClassifier
bagging_logreg = BaggingClassifier(
    estimator = best_estimators['LogisticRegression'],
    n_estimators = 50,
    random_state = 42,
    n_jobs = -1
)

ensemble_res('Bagging Logistic Regression', bagging_logreg)

[Bagging Logistic Regression] Accuracy: 0.8608 (+/- 0.0235)


In [6]:
#Bagging_classifier
from sklearn.ensemble import BaggingClassifier
bagging_svc = BaggingClassifier(
    estimator = best_estimators['SVC'],
    n_estimators = 50,
    random_state = 42,
    n_jobs = -1
)

ensemble_res('Bagging SVC', bagging_svc)

[Bagging SVC] Accuracy: 0.8687 (+/- 0.0185)


In [7]:
#AdaBoost and Stohastic Gradient Boost
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier

adaboost = AdaBoostClassifier(
    n_estimators = 100,
    learning_rate = 0.05,
    random_state = 42
)
ensemble_res('AdaBoost', adaboost)

sgb = GradientBoostingClassifier(
    n_estimators = 100,
    subsample = 0.8,
    max_features = 'sqrt',
    random_state = 42
)
ensemble_res('Stohastic Gradient Boost', sgb)

[AdaBoost] Accuracy: 0.8563 (+/- 0.0115)
[Stohastic Gradient Boost] Accuracy: 0.8754 (+/- 0.0132)


In [8]:
results_ensembles = pd.DataFrame(ensemble_results).sort_values(by = 'Accuracy', ascending = False)
results_ensembles

,Ensemble,Accuracy,Std
0,Voting hard,0.876536,0.014267
1,Voting soft,0.875413,0.012073
7,Stohastic Gradient Boost,0.875394,0.013212
2,Weighted averaging Top Boosters,0.874289,0.014978
3,Stacking Ridge Meta-model,0.874283,0.013234
5,Bagging SVC,0.868665,0.018457
4,Bagging Logistic Regression,0.860812,0.023502
6,AdaBoost,0.856349,0.011457
